# Baseline v7 — HyDE Combine (Query + Hypothetical Document)

**Pipeline:** V6 bi-encoder + V6 cross-encoder (PhoBERT CE) + **Merge Query & HyDE Retrieval**

## Ý tưởng

Không thay thế query embedding mà **search song song** với cả hai:

```
User Query
   ├─ embed(query)   → FAISS top-k1
   │
   ├─ LLM → hyde_doc → embed(hyde_doc) → FAISS top-k2
   │
   └─ Merge & Deduplicate (union)
              ↓
         PhoBERT CE Rerank
              ↓
           Final Top-5
```

### Tại sao KHÔNG thay thế query?
> LLM có thể hallucinate (sinh ra điều luật sai). Nếu chỉ dùng HyDE embedding, kết quả sẽ lệch theo hallucination.
> Bằng cách merge, query gốc làm **tấm lưới an toàn** — đảm bảo recall không giảm.

| | v6 | v7 Replace (sai) | **v7 Merge (đúng)** |
|--|--|--|--|
| Recall@1 | 0.5975 | ? | ? |
| Recall@5 | 0.7771 | ? | ? |
| MRR@10 | 0.6780 | ? | ? |

## Cell 0 — Config & Imports

In [ ]:
import json, csv, time, random, gc
import numpy as np
import faiss
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── Paths ──
ROOT         = Path(".")
DATA_DIR     = ROOT / "data"
EVAL_DIR     = ROOT / "outputs" / "eval"
TMP_DIR      = ROOT / "outputs" / "tmp"
MDL_DIR      = ROOT / "outputs" / "models"

EVAL_QA_FILE  = EVAL_DIR / "eval_qa.jsonl"
FAISS_V4      = TMP_DIR  / "faiss_v4.index"
MAP_V4        = TMP_DIR  / "faiss_mapping_v4.jsonl"

# Models (từ V6)
FT_BI_PATH    = MDL_DIR / "legal_hf_finetuned" / "final"
CE_V6_PATH    = MDL_DIR / "cross_encoder_v6" / "saved_model"
if not CE_V6_PATH.exists():
    CE_V6_PATH = MDL_DIR / "cross_encoder_v6"

RERANK_CSV_V6 = EVAL_DIR / "rerank_metrics_v6.csv"
RERANK_CSV_V7 = EVAL_DIR / "rerank_metrics_v7.csv"

# ── HyDE config ──
HYDE_MODEL  = "Qwen/Qwen2.5-1.5B-Instruct"
HYDE_N      = 3      # số hypothetical docs (MultiHyDE)
TOP_K_EACH  = 25     # top-k per search (query + cada hyde -> tổng tối đa 25*(1+N) trước dedup)

# ── Eval config ──
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
CE_BATCH  = 32
random.seed(42)

print(f"torch      : {torch.__version__}")
print(f"Device     : {DEVICE}")
print(f"HyDE model : {HYDE_MODEL}")
print(f"HyDE N     : {HYDE_N} docs")
print(f"top_k each : {TOP_K_EACH}  → merged pool tối đa {TOP_K_EACH * (1 + HYDE_N)} docs")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path, max_rows=None):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows: break
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: errors += 1
    if errors: print(f"  ⚠ {errors} malformed lines in {Path(path).name}")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_hit(faiss_id, expected_citations, mapping):
    row = mapping[faiss_id]
    for ec in expected_citations:
        ci = ec.get("chunk_index", -2)
        if ci != -1 and row["chunk_index"] == ci: return True
        if (row["van_ban"] == ec.get("van_ban", "") and
            row["dieu"]    == ec.get("dieu",    "") and
            row["khoan"]   == ec.get("khoan",   "")): return True
    return False

def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

print("Utilities loaded ✓")

## Cell 2 — Load Bi-Encoder & FAISS

In [ ]:
print(f"Loading bi-encoder: {FT_BI_PATH}")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
print(f"Bi-encoder ✓ | dim={ft_bi.get_sentence_embedding_dimension()}")
print(f"FAISS ✓      | {index_v4.ntotal} vectors, {len(mapping_v4)} mapping")

## Cell 3 — Load LLM nhỏ (Qwen2.5-1.5B-4bit) để sinh HyDE

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

hyde_tok = AutoTokenizer.from_pretrained(HYDE_MODEL, trust_remote_code=True)
hyde_lm  = AutoModelForCausalLM.from_pretrained(
    HYDE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
hyde_lm.config.use_cache = False
print(f"HyDE LM loaded: {HYDE_MODEL}")
print(f"Device: {next(hyde_lm.parameters()).device}")

## Cell 4 — HyDE Generation

In [ ]:
HYDE_SYSTEM = (
    "Bạn là trợ lý pháp lý. Khi nhận câu hỏi về pháp luật Việt Nam, "
    "hãy viết một đoạn điều khoản pháp lý ngắn (2-4 câu) như trong văn bản quy phạm pháp luật "
    "có thể trả lời câu hỏi đó. Viết tiếng Việt, văn phong pháp lý. "
    "Không cần chính xác tuyệt đối — đúng cấu trúc và ngữ nghĩa là đủ. "
    "Chỉ viết đoạn điều khoản, không giải thích."
)

@torch.no_grad()
def _gen_one(query: str, temperature: float = 0.7) -> str:
    msgs = [
        {"role": "system", "content": HYDE_SYSTEM},
        {"role": "user",   "content": f"Câu hỏi: {query}"},
    ]
    enc = hyde_tok.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    # handle both dict and tensor return
    if isinstance(enc, torch.Tensor):
        input_ids = enc.to(hyde_lm.device)
        attn_mask = None
    else:
        input_ids = enc["input_ids"].to(hyde_lm.device)
        attn_mask = enc.get("attention_mask")
        if attn_mask is not None:
            attn_mask = attn_mask.to(hyde_lm.device)
    prompt_len = input_ids.shape[-1]

    out = hyde_lm.generate(
        input_ids=input_ids,
        attention_mask=attn_mask,
        max_new_tokens=120,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=hyde_tok.eos_token_id,
    )
    return hyde_tok.decode(out[0][prompt_len:], skip_special_tokens=True).strip()


def generate_hyde_docs(query: str, n: int = HYDE_N) -> list[str]:
    """Sinh n hypothetical documents với temperature khác nhau."""
    temps = [0.5, 0.7, 0.9, 0.6, 0.8][:n]
    return [_gen_one(query, t) for t in temps]


# ── Test ──
test_q = "Thẩm quyền của UBND cấp xã trong quản lý đất đai là gì?"
test_hydes = generate_hyde_docs(test_q, n=1)
print(f"Query : {test_q}")
print(f"HyDE  : {test_hydes[0][:250]}...")

## Cell 5 — Load Cross-Encoder V6 (PhoBERT CE)

In [ ]:
print(f"Loading CE: {CE_V6_PATH}")
ce_model = CrossEncoder(str(CE_V6_PATH), max_length=256, device=DEVICE)
print("Cross-encoder V6 (PhoBERT CE) loaded ✓")

## Cell 6 — Hàm Combine Retrieve

**Thuật toán:**
```
embed(query)      → search top-k → set A
embed(hyde_1)     → search top-k → set B
embed(hyde_2)     → search top-k → set C   (MultiHyDE)
embed(hyde_3)     → search top-k → set D

A ∪ B ∪ C ∪ D  (dedup by faiss_id)
      ↓
    Rerank
      ↓
    Top-5
```

In [ ]:
def search_one(emb: np.ndarray, k: int) -> list:
    """Search FAISS, trả về list faiss_ids."""
    emb = emb.reshape(1, -1).astype("float32")
    _, ids = index_v4.search(emb, k)
    return [i for i in ids[0].tolist() if i >= 0]


def combine_retrieve(query: str, n_hyde: int = HYDE_N, top_k: int = TOP_K_EACH) -> tuple:
    """
    Trả về (merged_ids, hyde_docs):
    - merged_ids: list faiss_id sau dedup, dùng để rerank
    - hyde_docs: list các hypothetical docs (để debug)
    """
    # 1. Embed query gốc
    q_emb  = ft_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")[0]
    ids_q  = search_one(q_emb, top_k)

    # 2. Sinh N HyDE docs + embed từng cái
    hyde_docs = generate_hyde_docs(query, n=n_hyde)
    ids_all   = list(ids_q)  # bắt đầu từ query results

    for doc in hyde_docs:
        h_emb = ft_bi.encode([doc], normalize_embeddings=True,
                              convert_to_numpy=True).astype("float32")[0]
        ids_h = search_one(h_emb, top_k)
        # Merge: thêm vào nếu chưa có (dedup by faiss_id)
        seen = set(ids_all)
        for fid in ids_h:
            if fid not in seen:
                ids_all.append(fid)
                seen.add(fid)

    return ids_all, hyde_docs


print(f"combine_retrieve defined ✓")
print(f"Pool size tối đa: {TOP_K_EACH} (query) + {TOP_K_EACH} × {HYDE_N} (hyde) = {TOP_K_EACH * (1 + HYDE_N)} (trước dedup)")

## Cell 7 — Evaluate: V6 vs V7 Combine Query+HyDE

So sánh 3 cấu hình:
- **V6**: embed(query) → top-50 → CE rerank
- **V7 Single**: Combine query + 1 HyDE doc → CE rerank  
- **V7 Multi**: Combine query + 3 HyDE docs → CE rerank

In [ ]:
eval_qa = load_jsonl(EVAL_QA_FILE)
print(f"Eval QA: {len(eval_qa)} questions")

r_v6     = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}
r_v7_s1  = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}  # Single HyDE (N=1)
r_v7_mN  = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}  # Multi  HyDE (N=3)

pool_sizes = []  # theo dõi kích thước pool sau merge

def rerank_ids(query, candidate_ids):
    """Rerank list faiss_id bằng CE, trả về list đã sort."""
    cands   = [(mapping_v4[i]["passage"], i) for i in candidate_ids if 0 <= i < len(mapping_v4)]
    if not cands:
        return []
    rscores = ce_model.predict([[query, c[0]] for c in cands], batch_size=CE_BATCH)
    ranked  = sorted(zip(rscores, [c[1] for c in cands]), reverse=True)
    return [r[1] for r in ranked]

def score_result(r_ids, result_dict, ec):
    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        result_dict[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in r_ids[:k]) else 0)
    mrr = 0.0
    for rank, i in enumerate(r_ids[:10], 1):
        if is_hit(i, ec, mapping_v4): mrr = 1.0/rank; break
    result_dict["MRR@10"].append(mrr)

t0 = time.time()

for item in tqdm(eval_qa, desc="Evaluate V7"):
    query = item["query"]
    ec    = item["expected_citations"]

    # ── V6: query only ──
    q_emb   = ft_bi.encode([query], normalize_embeddings=True,
                            convert_to_numpy=True).astype("float32")[0]
    ids_v6  = search_one(q_emb, k=50)
    r_ids_v6 = rerank_ids(query, ids_v6)
    score_result(r_ids_v6, r_v6, ec)

    # ── V7 Single (query + 1 HyDE) ──
    ids_s1, _ = combine_retrieve(query, n_hyde=1, top_k=TOP_K_EACH)
    r_ids_s1  = rerank_ids(query, ids_s1)
    score_result(r_ids_s1, r_v7_s1, ec)
    pool_sizes.append(len(ids_s1))

    # ── V7 Multi (query + 3 HyDE) ──
    ids_mN, hydes = combine_retrieve(query, n_hyde=HYDE_N, top_k=TOP_K_EACH)
    r_ids_mN      = rerank_ids(query, ids_mN)
    score_result(r_ids_mN, r_v7_mN, ec)

elapsed = time.time() - t0
print(f"\nEval done in {elapsed:.1f}s  |  {elapsed/len(eval_qa):.2f}s/query")
print(f"Avg pool size (Single): {sum(pool_sizes)/len(pool_sizes):.1f} docs trước rerank")

print("\n" + "="*92)
print(f"  {'Metric':<10} {'V6 (query only)':<18} {'V7 Single(q+1H)':<20} {'V7 Multi(q+3H)':<18} Best")
print("="*92)
for k, key in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]:
    b  = avg(r_v6[k])
    s1 = avg(r_v7_s1[k])
    mN = avg(r_v7_mN[k])
    best = max(b, s1, mN)
    tag  = "← V6" if best==b else ("← S1" if best==s1 else "← MN")
    print(f"  {key:<10} {b:<18.4f} {s1:<20.4f} {mN:<18.4f} {best:.4f} {tag}")
print("="*92)

## Cell 8 — Lưu kết quả CSV

In [ ]:
# Chọn variant tốt nhất của V7
v6_r = {m: avg(r_v6[k]) for k,m in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]}
s1_r = {m: avg(r_v7_s1[k]) for k,m in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]}
mN_r = {m: avg(r_v7_mN[k]) for k,m in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]}

print("\n" + "="*72)
print(f"  {'Metric':<10} {'V6':<12} {'V7 Single':<14} {'V7 Multi':<12} {'Δ(Multi-V6)'}")
print("="*72)
for metric in ["Recall@1", "Recall@3", "Recall@5", "MRR@10"]:
    b  = v6_r[metric]
    s  = s1_r[metric]
    m  = mN_r[metric]
    d  = m - b
    sign = "+" if d >= 0 else ""
    print(f"  {metric:<10} {b:<12.4f} {s:<14.4f} {m:<12.4f} {sign}{d:.4f}")
print("="*72)

rows_v7 = [
    {"metric": "Recall@1", "v6": v6_r["Recall@1"], "v7_single_hyde": s1_r["Recall@1"], "v7_multi_hyde": mN_r["Recall@1"]},
    {"metric": "Recall@3", "v6": v6_r["Recall@3"], "v7_single_hyde": s1_r["Recall@3"], "v7_multi_hyde": mN_r["Recall@3"]},
    {"metric": "Recall@5", "v6": v6_r["Recall@5"], "v7_single_hyde": s1_r["Recall@5"], "v7_multi_hyde": mN_r["Recall@5"]},
    {"metric": "MRR@10",   "v6": v6_r["MRR@10"],   "v7_single_hyde": s1_r["MRR@10"],   "v7_multi_hyde": mN_r["MRR@10"]},
]
with open(RERANK_CSV_V7, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["metric","v6","v7_single_hyde","v7_multi_hyde"])
    w.writeheader(); w.writerows(rows_v7)
print(f"\nSaved → {RERANK_CSV_V7} ✓")

## Cell 9 — Latency Analysis

In [ ]:
test_qs = [item["query"] for item in eval_qa[:10]]

# V6
t0 = time.time()
for q in test_qs:
    e = ft_bi.encode([q], normalize_embeddings=True, convert_to_numpy=True)
    search_one(e[0], k=50)
v6_ms = (time.time()-t0)/len(test_qs)*1000

# V7 Single
t0 = time.time()
for q in test_qs:
    combine_retrieve(q, n_hyde=1, top_k=TOP_K_EACH)
v7s_ms = (time.time()-t0)/len(test_qs)*1000

# V7 Multi
t0 = time.time()
for q in test_qs:
    combine_retrieve(q, n_hyde=HYDE_N, top_k=TOP_K_EACH)
v7m_ms = (time.time()-t0)/len(test_qs)*1000

print("── Latency (retrieval step only, avg per query) ──")
print(f"  V6  (query only)      : {v6_ms:>7.1f} ms")
print(f"  V7  (query + 1 HyDE)  : {v7s_ms:>7.1f} ms  ({v7s_ms/v6_ms:.1f}x)")
print(f"  V7  (query + {HYDE_N} HyDE)  : {v7m_ms:>7.1f} ms  ({v7m_ms/v6_ms:.1f}x)")
print(f"\n  Latency overhead (HyDE lần đầu bào gồm LLM generation time)")